In [25]:
from transformers import AutoModelForMaskedLM, AutoTokenizer
import anndata as ad
import numpy as np
import requests
import os

In [26]:
# Load model and tokenizer
model = AutoModelForMaskedLM.from_pretrained("theislab/Nicheformer", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("theislab/Nicheformer", trust_remote_code=True)

In [27]:
# 2. Download the file via streaming (safest for large files)
url = "https://github.com/BiocFM/nicheR/blob/a10236f725ef6f5c14f2dd63fa32ed6d1b7b6bdf/inst/scripts/data/xenium_mean_script.npy"
local_filename = "xenium_mean_script.npy"
if not os.path.exists(local_filename):
    print(f"Downloading file from {url}...")
    response = requests.get(url, stream=True, allow_redirects=True)
    response.raise_for_status()  # Check for HTTP errors

    with open(local_filename, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print(f"Saved to {local_filename}")

In [28]:
# Set technology mean for HF tokenizer
technology_mean_path = local_filename
technology_mean = np.load(technology_mean_path, allow_pickle=True)
tokenizer._load_technology_mean(technology_mean)

Using provided technology mean array with shape (20310,)


In [29]:
# 2. Download the file via streaming (safest for large files)
url = "https://github.com/BiocFM/nicheR/blob/main/inst/extdata/spe_converted_subset_200_iter_1.h5ad"
local_filename = "spe_converted_subset_200_iter_1.h5ad"
if not os.path.exists(local_filename):
    print(f"Downloading file from {url}...")
    response = requests.get(url, stream=True, allow_redirects=True)
    response.raise_for_status()  # Check for HTTP errors

    with open(local_filename, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print(f"Saved to {local_filename}")

In [ ]:
# Load your single-cell data
adata = ad.read_h5ad("spe_converted_subset_200_iter_1.h5ad")
adata.X = adata.layers["counts"]
adata

In [42]:

# Tokenize the data
inputs = tokenizer(adata)
print(inputs['input_ids'].shape)
print(inputs['attention_mask'].shape)

torch.Size([200, 1500])
torch.Size([200, 1500])


In [ ]:

# Get embeddings
embeddings = model.get_embeddings(
              input_ids=inputs["input_ids"],
              attention_mask=inputs["attention_mask"],
              layer=-1,
              with_context=False
            )


In [38]:
embeddings.shape

torch.Size([200, 512])